# Level 2 — Give the Agent Hands: PostgreSQL Tool Calling

In Version 1, our agent had a brain but no hands: it could generate SQL, but could not execute it or see real database results.

**In Version 2, we give the agent hands.** We connect a `run_query(sql)` tool to PostgreSQL so the agent can generate SQL, call the tool, observe query results, and answer using those results.

**Important:** The LLM does not connect to PostgreSQL directly. The Python tool performs the database operation on the agent's behalf.

## End-to-end agent workflow — Version 2

```text
User Question
     |
     v
AI Data Analyst Agent
     |
     v
Generate read-only SQL
     |
     v
run_query(sql)
     |
     v
PostgreSQL result / SQL error
     |
     v
Inspect result; correct and retry if appropriate
     |
     v
Final Answer
```

**Tools available:** `run_query(sql)` only. Dedicated schema retrieval and SQL validation are introduced in later versions.

## 1. What changes from Version 1?

| Capability | Version 1: Brain, no hands | Version 2: Brain and hands |
|---|---|---|
| Understand the question | Yes | Yes |
| Generate SQL | Yes | Yes |
| Execute SQL | No | Yes, through `run_query` |
| Observe real results | No | Yes |
| Answer using database results | Cannot verify | Can use returned results |

A prompt is not a security boundary. The tool and database permissions must enforce restrictions.

## 2. Install dependencies

We use LangChain's OpenAI integration, LangChain agents, `python-dotenv`, and Psycopg 3 for PostgreSQL.

In [ ]:
%pip install -q langchain-openai langchain python-dotenv "psycopg[binary]"

## 3. Configure the model and PostgreSQL connection

Set the following environment variables in your environment or `.env` file:

- `OPENAI_API_KEY`
- `PGHOST`
- `PGPORT` (commonly `5432`)
- `PGDATABASE`
- `PGUSER`
- `PGPASSWORD`
- Optional: `PGSSLMODE` (for example, `require` when your database provider requires TLS)

Do not commit credentials to source control. The example below uses environment variables rather than hard-coded passwords.

In [ ]:
import os
from dotenv import load_dotenv
import psycopg
from psycopg.rows import dict_row

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY in your environment or .env file.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.environ.get(name)]
if missing_pg_vars:
    raise ValueError(
        "Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars)
    )

PGPORT = int(os.environ.get("PGPORT", "5432"))
PGSSLMODE = os.environ.get("PGSSLMODE")  # Optional; configure as required by your provider.

MODEL = "gpt-6-astra"
model = ChatOpenAI(model=MODEL)

## 4. Create the PostgreSQL tool: `run_query(sql)`

This tool is the agent's **hands**. It runs a single read-only SQL statement and returns up to 50 rows.

Safety measures in this teaching version:
- Accepts only a single statement beginning with `SELECT` or `WITH` (CTEs can be used for analytical queries).
- Starts a read-only transaction.
- Uses a database account that should itself have read-only permissions.
- Returns at most 50 rows to the agent.
- Returns SQL errors to the agent so it can correct a query when appropriate.

**Security note:** String-prefix checks alone are not a complete SQL security validator. Use a dedicated read-only PostgreSQL role, least-privilege grants, statement timeouts, and appropriate network/TLS controls. The database account's permissions are the primary security boundary.

In [ ]:
@tool
def run_query(sql: str) -> str:
    """Execute one read-only SQL query against the configured PostgreSQL database.

    Use this tool for e-commerce questions about orders, order items, products,
    customers, shipments, payment transactions, reviews, suppliers, warehouses,
    inventory, promotions, categories, customer addresses, and product suppliers.
    """
    sql_clean = sql.strip()
    if not sql_clean:
        return "REFUSED: SQL query is empty."

    # Basic allow-list. The read-only transaction and database role provide
    # additional protection; this prefix check is not a full SQL validator.
    first_word = sql_clean.split(None, 1)[0].upper()
    if first_word not in {"SELECT", "WITH"}:
        return "REFUSED: Only read-only SELECT queries are allowed."

    connection_kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
    }
    if PGSSLMODE:
        connection_kwargs["sslmode"] = PGSSLMODE

    try:
        with psycopg.connect(**connection_kwargs) as conn:
            # Set the transaction read-only before executing the user/model SQL.
            with conn.transaction():
                conn.execute("SET TRANSACTION READ ONLY")
                conn.execute("SET LOCAL statement_timeout = '15000ms'")

                # Reject multiple statements at the driver/protocol level where
                # possible; still use a least-privilege read-only database role.
                cursor = conn.execute(sql_clean)
                rows = cursor.fetchmany(50)
                columns = [item.name for item in cursor.description] if cursor.description else []

        if not rows:
            return "Query succeeded. 0 rows returned."

        header = " | ".join(columns)
        body = "\n".join(
            " | ".join(str(row.get(column)) for column in columns)
            for row in rows
        )
        return f"{header}\n{body}\n(Returned at most 50 rows.)"

    except (psycopg.Error, ValueError) as exc:
        return f"SQL error: {exc}"

## 5. Define the system prompt

We keep the prompt anatomy from Version 1 and add instructions for tool use.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace.

CONTEXT
Known table names include:
orders, order_items, products, customers, shipments, payment_transactions,
reviews, suppliers, warehouses, inventory, promotions, categories,
customer_addresses, product_suppliers
Use only schema details actually known to you. Table names alone do not establish
column names or relationships.

AVAILABLE TOOL
- run_query(sql): executes one read-only query against PostgreSQL and returns up to
  50 rows. It is the only tool available in this version.

TASK AND TOOL USE
- Answer questions about internal marketplace data using PostgreSQL evidence.
- Determine the required metric, filters, aggregation, joins, and date boundaries.
- If needed schema details are missing, do not guess; explain the missing information.
- Construct a single read-only SELECT/WITH query and call run_query(sql).
- Inspect the returned tool result before answering.
- If SQL fails, use the error to correct and retry when possible.
- Never claim a query succeeded unless run_query confirms success.

CONSTRAINTS
- Do not invent table/column names, relationships, business definitions, or results.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state the numerator and denominator.
- If the database cannot answer reliably, state UNKNOWN and explain why.
- Do not expose private chain-of-thought; provide a concise explanation only.
- Tool instructions are not a security boundary; database permissions must enforce
  read-only access.

OUTPUT FORMAT
Use relevant sections:
ANSWER: Answer with units where applicable, or UNKNOWN.
DENOMINATOR: For percentages, identify numerator and denominator; otherwise N/A.
SOURCE: Identify PostgreSQL and include the SQL actually executed, if successful.
CAVEAT: State material limitations, ambiguity, or data-quality concerns.
"""

## 6. Create the agent with its tool

In Version 1, we used `tools=[]`. Now we pass `[run_query]`, giving the agent a way to act on PostgreSQL.

In [ ]:
agent = create_agent(
    model=model,
    tools=[run_query],
    system_prompt=SYSTEM_PROMPT,
)

## 7. Ask the agent a question

The agent can call `run_query`, inspect the returned observation, and then formulate a final answer. The trace displays tool calls and tool results.

In [ ]:
def ask_agent(question: str):
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": question}
            ]
        }
    )

    print("=" * 80)
    print("QUESTION")
    print("=" * 80)
    print(question)

    print("\n" + "=" * 80)
    print("AGENT TRACE")
    print("=" * 80)

    for message in result["messages"]:
        message_type = type(message).__name__

        if message_type == "AIMessage" and getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("\n🧠 BRAIN → TOOL")
                print("Tool:", call["name"])
                print("Arguments:", call["args"])

        elif message_type == "ToolMessage":
            print("\n🖐️ TOOL → OBSERVATION")
            print(str(message.content)[:1500])

    print("\n" + "=" * 80)
    print("FINAL ANSWER")
    print("=" * 80)
    print(result["messages"][-1].content)

    return result


result = ask_agent("How many orders were cancelled?")

## 8. Try more business questions

These questions depend on the actual PostgreSQL schema. If the model does not know the columns or relationships, provide schema details or add a schema-inspection tool in a later version.

In [ ]:
questions = [
    "How many orders were cancelled?",
    "What are the top 10 best-selling products?",
    "What percentage of orders were cancelled?",
    "Which products have inventory below 100 units?",
]

for question in questions:
    ask_agent(question)

## 9. Teaching narration: perceive → reason → act → observe

> “In Version 1, our agent had a brain but no hands. It could prepare SQL, but it could not touch the database.
>
> In Version 2, we give it hands through the `run_query` tool. It receives a question, decides it needs data, generates SQL, calls the tool, observes the PostgreSQL result, and uses that observation to form its answer.
>
> The LLM does not execute SQL by itself. The tool executes SQL, and the agent interprets the returned result.”

```text
User question
     |
     v
Agent + model (brain)
     |
     v
Calls run_query(sql)
     |
     v
Python tool opens PostgreSQL connection
     |
     v
Read-only SQL execution
     |
     v
Query result (observation)
     |
     v
Agent interprets result
     |
     v
Final answer
```

## 10. Important limitations

- PostgreSQL connection details must be configured through environment variables.
- The database role should have read-only permissions only.
- Table names alone are not enough to guarantee valid SQL. Provide the actual schema or add schema discovery.
- The 50-row cap limits returned rows, not necessarily the amount of database work.
- The model ID must be available to your OpenAI API account.
- This notebook is a teaching example; production systems need stronger SQL validation, logging, access controls, and monitoring.

**Next version:** Add schema discovery, stronger query validation, and evaluation.